# Phase 2 — Welder Projection into PD Severity Space + Exposure Associations

**Prerequisite:** Run `phase1_pd_hy_model.ipynb` first to generate  
`pd_hy_binary_model.pkl` and `pd_hy_multiclass_model.pkl`.

**What this notebook does:**
- **Model B:** Applies the PD-trained H&Y classifier to welders — each welder  
  receives a predicted PD-like severity class and class probabilities.
- **Model C:** Tests whether predicted PD-like severity in welders is associated  
  with welding exposure variables (years, hours/day, PPE use, fume/vibration exposure).

**Framing:**  
> Welders are not assigned a clinical H&Y stage.  
> Instead, we ask: *which PD severity profile does each welder most resemble?*  
> And: *does resemblance to more severe PD patterns correlate with heavier exposure?*

> ⚠ Pilot study (n=16 welders). Results are exploratory and hypothesis-generating.


## 0. Setup


In [ ]:
%pip install pandas numpy scikit-learn matplotlib seaborn scipy openpyxl joblib -q
import pandas as pd, numpy as np, re, warnings, joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
warnings.filterwarnings('ignore')
print('Libraries loaded.')


## 1. Load Data and Trained Models

Load the WD (welders) sheet and the two saved models from Phase 1.


In [ ]:
# ── Data ────────────────────────────────────────────────────────────────────
try:
    from google.colab import files
    print('Upload PD_WELDERS RAW Long Data-2.xlsx')
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
except ImportError:
    DATA_PATH = 'PD_WELDERS RAW Long Data-2.xlsx'

wd_raw = pd.ExcelFile(DATA_PATH).parse('WD')
pd_raw = pd.ExcelFile(DATA_PATH).parse('PD')
print(f'WD sheet: {wd_raw.shape[0]} rows × {wd_raw.shape[1]} columns')

# ── Models (from phase1) ────────────────────────────────────────────────────
try:
    bin_model = joblib.load('pd_hy_binary_model.pkl')
    mc_model  = joblib.load('pd_hy_multiclass_model.pkl')
    print(f'Binary model loaded  : {bin_model["model_name"]}')
    print(f'Multi-class model    : {mc_model["model_name"]}')
except FileNotFoundError:
    print('⚠ Model files not found. Run phase1_pd_hy_model.ipynb first')
    print('  and ensure pd_hy_binary_model.pkl / pd_hy_multiclass_model.pkl')
    print('  are in the same directory as this notebook.')


## 2. Extract Welder Features

Extract balance features and exposure variables from the WD sheet.


In [ ]:
def parse_fall_wd(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    return 0 if s in ['no','nil','none','0',''] else (1 if s in ['yes','y'] else np.nan)

def encode_exposure(v):
    """Occasional < Regular < Frequent → 0/1/2/3."""
    if pd.isna(v): return np.nan
    s = str(v).lower()
    if 'never' in s or 'none' in s: return 0
    if 'occasional' in s or 'less' in s: return 1
    if 'regular' in s or 'sometimes' in s or 'moderate' in s: return 2
    if 'frequent' in s or 'always' in s or 'high' in s: return 3
    return np.nan

def encode_ppe(v):
    """Never/Sometimes/Always → 0/1/2."""
    if pd.isna(v): return np.nan
    s = str(v).lower()
    if 'never' in s: return 0
    if 'sometime' in s or 'occasional' in s: return 1
    if 'always' in s or 'regular' in s: return 2
    return np.nan

records = []
for _, r in wd_raw.iterrows():
    rec = dict(
        ID        = r.get("Participant's Name", ''),
        Age       = pd.to_numeric(r.get('Age'), errors='coerce'),
        BBS       = pd.to_numeric(r.get('BBS'), errors='coerce'),
        Mini      = pd.to_numeric(r.get('MINI-BEST SCORE'), errors='coerce'),
        FES       = pd.to_numeric(r.get('FES'), errors='coerce'),
        # Exposure variables
        WeldYrs   = pd.to_numeric(r.get('Total Years in Welding'), errors='coerce'),
        HrsPerDay = pd.to_numeric(r.get('Work Hours per Day'), errors='coerce'),
        FallHist  = parse_fall_wd(r.get('History of Fall')),
        FumeExp   = encode_exposure(r.get(
            next((c for c in wd_raw.columns if 'fume' in str(c).lower()), None))),
        VibExp    = encode_exposure(r.get(
            next((c for c in wd_raw.columns if 'vibrat' in str(c).lower()), None))),
        NoiseExp  = encode_exposure(r.get(
            next((c for c in wd_raw.columns if 'noise' in str(c).lower() and 'exposure' in str(c).lower()), None))),
        RespPPE   = encode_ppe(r.get(
            next((c for c in wd_raw.columns if 'respirat' in str(c).lower() and 'ppe' in str(c).lower()
                  or ('respiratory' in str(c).lower() and 'protect' in str(c).lower())), None))),
    )
    records.append(rec)

wd_df = pd.DataFrame(records)
print(wd_df[['ID','Age','BBS','Mini','FES','WeldYrs','HrsPerDay','FallHist']].to_string(index=False))
print(f'\nn={len(wd_df)}  Age: {wd_df.Age.mean():.1f}±{wd_df.Age.std(ddof=1):.1f}')
print(f'WeldYrs: {wd_df.WeldYrs.min():.0f}–{wd_df.WeldYrs.max():.0f} yrs')


## 3. W1–W5 Welding Exposure Stage Assignment *(Exploratory)*

A preliminary staging schema based on years of welding exposure,  
analogous to the H&Y staging concept for PD.

| Stage | Years | Interpretation |
|-------|-------|----------------|
| W1 | 0–<10 | Early exposure |
| W2 | 10–<20 | Moderate exposure |
| W3 | 20–<30 | Significant exposure |
| W4 | 30–<40 | Heavy exposure |
| W5 | ≥40 | Very heavy exposure |

> ⚠ This staging is exploratory and not clinically validated.


In [ ]:
def w_stage(yrs):
    if pd.isna(yrs): return np.nan
    if yrs < 10:  return 1
    if yrs < 20:  return 2
    if yrs < 30:  return 3
    if yrs < 40:  return 4
    return 5

wd_df['W_Stage'] = wd_df['WeldYrs'].apply(w_stage)
print('W-Stage distribution:')
print(wd_df['W_Stage'].value_counts().sort_index().to_dict())
print()
print(wd_df[['ID','WeldYrs','W_Stage','BBS','Mini','FES']].to_string(index=False))


## 4. Model B — Welder Projection into PD Severity Space

Apply the PD-trained models to all 16 welders.  
Each welder receives:
- A **predicted binary class** (Early-like / Late-like)
- A **PD-like severity score** (weighted probability: 1×P(I) + 2×P(II) + 3×P(III) + 4×P(IV))
- **Class probabilities** for each H&Y stage

**Interpretation:** This is not a clinical diagnosis. It describes which PD severity  
profile each welder's balance pattern most resembles.


In [ ]:
FEAT = ['BBS','Mini','FES']
wd_clean = wd_df.dropna(subset=FEAT).copy()

# ── Binary projection ───────────────────────────────────────────────────────
X_wd = wd_clean[FEAT].values.astype(float)
X_wd_bin_sc  = bin_model['scaler'].transform(X_wd)
X_wd_mc_sc   = mc_model['scaler'].transform(X_wd)

wd_clean['Pred_Binary']     = bin_model['clf'].predict(X_wd_bin_sc)
wd_clean['Pred_Binary_Lbl'] = wd_clean['Pred_Binary'].map(
    {0:'Early-like (I-II)', 1:'Late-like (III-IV)'})

# ── Multi-class stage probabilities ─────────────────────────────────────────
mc_proba  = mc_model['clf'].predict_proba(X_wd_mc_sc)
mc_stages = mc_model['classes']
for i, s in enumerate(mc_stages):
    wd_clean[f'P_Stage{s}'] = mc_proba[:, i]

wd_clean['Pred_Stage'] = mc_model['clf'].predict(X_wd_mc_sc)

# ── PD-like severity score (weighted mean of stage probabilities) ────────────
wd_clean['PD_Severity_Score'] = sum(
    s * mc_proba[:, i] for i, s in enumerate(mc_stages)
)

# ── Print results ────────────────────────────────────────────────────────────
print('=' * 75)
print('MODEL B — Welder Projection Results')
print('=' * 75)
display_cols = ['ID','WeldYrs','W_Stage','BBS','Mini','FES',
                'Pred_Binary_Lbl','Pred_Stage','PD_Severity_Score']
display_cols = [c for c in display_cols if c in wd_clean.columns]
print(wd_clean[display_cols].to_string(index=False))
print()
print('Binary projection summary:')
print(wd_clean['Pred_Binary_Lbl'].value_counts().to_string())
print()
print('Predicted stage distribution:')
print(wd_clean['Pred_Stage'].value_counts().sort_index().to_string())
print(f'\nMean PD-like severity score: {wd_clean["PD_Severity_Score"].mean():.2f} '
      f'(scale 1=Stage I to 4=Stage IV)')


## 5. Visualise Welder Projection


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: PD severity score distribution
axes[0].hist(wd_clean['PD_Severity_Score'], bins=8,
             color='steelblue', edgecolor='k', alpha=0.8)
axes[0].axvline(wd_clean['PD_Severity_Score'].mean(), color='red',
                linestyle='--', label=f'Mean={wd_clean["PD_Severity_Score"].mean():.2f}')
axes[0].set_xlabel('PD-like Severity Score (1–4)')
axes[0].set_ylabel('Number of Welders')
axes[0].set_title('Welder Distribution in\nPD Severity Space')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Plot 2: Stacked probability bars per welder
colors_stage = {1:'#3498db', 2:'#2ecc71', 3:'#f39c12', 4:'#e74c3c'}
bottoms = np.zeros(len(wd_clean))
for s in mc_stages:
    col = f'P_Stage{s}'
    if col in wd_clean.columns:
        axes[1].bar(range(len(wd_clean)), wd_clean[col],
                    bottom=bottoms, label=f'Stage {s}',
                    color=colors_stage[s], alpha=0.85)
        bottoms += wd_clean[col].values
axes[1].set_xticks(range(len(wd_clean)))
axes[1].set_xticklabels(wd_clean['ID'].values, rotation=45, ha='right', fontsize=7)
axes[1].set_ylabel('Probability')
axes[1].set_title('Stage Probability per Welder\n(stacked)')
axes[1].legend(loc='upper right', fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

# Plot 3: Severity score vs welding years
sub = wd_clean.dropna(subset=['WeldYrs','PD_Severity_Score'])
axes[2].scatter(sub['WeldYrs'], sub['PD_Severity_Score'],
                c=sub['W_Stage'], cmap='YlOrRd', s=80, edgecolors='k',
                linewidths=0.6, alpha=0.9)
if len(sub) >= 3:
    m, b, *_ = stats.linregress(sub['WeldYrs'], sub['PD_Severity_Score'])
    xl = np.linspace(sub['WeldYrs'].min(), sub['WeldYrs'].max(), 100)
    rho, pval = stats.spearmanr(sub['WeldYrs'], sub['PD_Severity_Score'])
    axes[2].plot(xl, m*xl+b, 'r--', linewidth=1.8)
    sig = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else 'ns'
    axes[2].set_title(f'Severity Score vs Welding Years\nρ={rho:+.3f} ({sig})')
axes[2].set_xlabel('Total Years in Welding')
axes[2].set_ylabel('PD-like Severity Score')
axes[2].grid(alpha=0.3)

plt.suptitle('Welder Projection into PD Motor-Balance Severity Space\n'
             '(n=16 welders | PD model trained on n=14 PD patients)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figure4_welder_projection.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Model C — Association Analysis: Predicted Severity vs Exposure Variables

Test whether welders who resemble more severe PD patterns (higher PD-like severity score)  
also have heavier welding exposure. This is the key clinical question.

Using **Spearman ρ** (non-parametric, appropriate for ordinal/non-normal variables).


In [ ]:
print('=' * 68)
print('MODEL C — Predicted PD-like Severity vs Exposure Variables')
print('(Spearman correlation, n=16 welders)')
print('=' * 68)

exposure_vars = [
    ('WeldYrs',   'Total Years in Welding'),
    ('HrsPerDay', 'Hours per Day'),
    ('FumeExp',   'Fume Exposure Intensity'),
    ('VibExp',    'Vibration Exposure Intensity'),
    ('NoiseExp',  'Noise Exposure Intensity'),
    ('RespPPE',   'Respiratory PPE Use'),
    ('FallHist',  'History of Falls'),
    ('W_Stage',   'W-Stage (exposure stage)'),
]

assoc_results = []
print(f'{"Variable":35s}  {"n":>4s}  {"rho":>7s}  {"p":>8s}  Sig')
print('-' * 65)
for var, label in exposure_vars:
    if var not in wd_clean.columns: continue
    sub = wd_clean.dropna(subset=[var,'PD_Severity_Score'])
    if len(sub) < 5: continue
    rho, pval = stats.spearmanr(sub[var], sub['PD_Severity_Score'])
    sig = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else 'ns'
    print(f'  {label:33s}  {len(sub):4d}  {rho:+.3f}    {pval:.4f}  {sig}')
    assoc_results.append(dict(var=var, label=label, rho=rho, pval=pval, n=len(sub), sig=sig))

print()
print('* p<0.05  ** p<0.01  *** p<0.001  ns = not significant')


## 7. Raw Balance Scales vs Exposure Variables (Welders only)

Spearman correlations between raw balance scores and exposure variables,  
to show which individual scales drive any association found in Model C.


In [ ]:
print('=' * 68)
print('Raw Balance Scale Correlations with Exposure (Welders, n=16)')
print('=' * 68)

for scale, worse in [('BBS','lower=worse'),('Mini','lower=worse'),('FES','higher=worse')]:
    print(f'\n  {scale} ({worse})')
    for var, label in exposure_vars:
        if var not in wd_clean.columns: continue
        sub = wd_clean.dropna(subset=[var, scale])
        if len(sub) < 5: continue
        rho, pval = stats.spearmanr(sub[var], sub[scale])
        sig = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else 'ns'
        print(f'    {label:33s}  ρ={rho:+.3f}  p={pval:.4f} ({sig})')


## 8. Phase 2 Results Summary


In [ ]:
print('=' * 68)
print('PHASE 2 — WELDER PROJECTION + EXPOSURE ASSOCIATIONS — SUMMARY')
print('=' * 68)
print()
print('MODEL B — Welder Projection into PD Severity Space:')
print(f'  n welders projected: {len(wd_clean)}')
print(f'  Binary projection:')
for lbl, cnt in wd_clean['Pred_Binary_Lbl'].value_counts().items():
    pct = cnt/len(wd_clean)*100
    print(f'    {lbl}: {cnt} ({pct:.0f}%)')
print(f'  Multi-class predicted stage distribution:')
for s, cnt in wd_clean['Pred_Stage'].value_counts().sort_index().items():
    pct = cnt/len(wd_clean)*100
    print(f'    Stage {s}: {cnt} ({pct:.0f}%)')
print(f'  Mean PD-like severity score: {wd_clean["PD_Severity_Score"].mean():.2f} ± '
      f'{wd_clean["PD_Severity_Score"].std(ddof=1):.2f}')
print()
print('MODEL C — Significant exposure associations (p < 0.05):')
sig_assoc = [r for r in assoc_results if r['pval'] < 0.05]
if sig_assoc:
    for r in sig_assoc:
        print(f'  {r["label"]}: ρ={r["rho"]:+.3f}  p={r["pval"]:.4f} {r["sig"]}')
else:
    print('  None reached p < 0.05 — findings are exploratory')
print()
print('INTERPRETATION')
print('  Welders are not assigned clinical H&Y stages.')
print('  Projection shows which PD severity profile each welder most resembles.')
print('  Any association with exposure is correlational, not causal.')
print()
print('LIMITATIONS')
print('  - PD reference model trained on n=14 only')
print('  - 30-year age gap (PD 71.7 vs Welder 41.8 yrs) is not fully controlled')
print('  - n=16 welders; all correlations should be replicated with larger n')
print('  - 7 welders missing fall history (excluded from fall-history association)')
print('  - W1-W5 staging is preliminary and not clinically validated')
